# Multi-provider model gateway with intelligent fallback routing

A high-traffic LLM app got hit by two hour-long OpenAI outages last quarter. The CTO wants graceful degradation: if OpenAI 5xxs or times out, fall back to Anthropic; if both fail, fall back to Together's Llama. You have 90 minutes to build the gateway: a thin Python service that wraps three providers, retries with exponential backoff, circuit-breaks on persistent failure, routes by latency p95 in normal operation, and emits a per-request cost label.

**Duration:** about 90 minutes of work in a 120-minute session window.

**Cost preamble.** About a buck. The gateway is free; you build it in the kernel. Provider tokens are the spend: ~30 OpenAI calls, ~20 Anthropic calls, ~10 Together calls plus a 100-request outage-simulation load test. A clean run lands near $1; a debug run can drift to $2 if the outage simulation runs twice.

In [ ]:
# NBVAL_SKIP
# Pinned dependencies. The three provider SDKs are the only externals;
# everything else (circuit breaker, cost map, p95 routing) is plain Python.

!pip install --quiet labrun-checks==0.7.0 anthropic==0.42.0 openai==1.54.0 together==1.3.0

In [ ]:
# Imports and per-lab constants. The provider names are the routing keys; the
# cost map is in cents per token (matches the brief's pricing table).

import atexit
import getpass
import json
import os
import random
import statistics
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from contextlib import contextmanager
from threading import Lock

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupAdapter,
    CleanupResource,
)

LAB_ID = "multi-provider-model-gateway"
LAB_TAG_VALUE = LAB_ID

TRACES_PATH = "/content/gateway_traces.json"
BREAKER_PATH = "/content/circuit_breaker_state.json"

PROVIDER_ORDER = ["openai", "anthropic", "together"]

# Per-1M-token prices in dollars; gateway converts to cents per request.
PRICE_MAP = {
    "openai": {"model": "gpt-4o-mini", "in_per_m": 0.15, "out_per_m": 0.60},
    "anthropic": {"model": "claude-haiku-4-5-20250930", "in_per_m": 0.80, "out_per_m": 4.00},
    "together": {"model": "meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo", "in_per_m": 0.20, "out_per_m": 0.20},
}

# Circuit breaker tuning.
BREAKER_FAILURE_THRESHOLD = 5
BREAKER_COOLDOWN_SEC = 30

# Tight per-provider deadline so a slow provider does not eat the latency
# budget for the next one in line.
PER_PROVIDER_TIMEOUT_SEC = 3.0

In [ ]:
# NBVAL_SKIP
# BYOK setup. Session token plus the three provider keys. Smoke-test each key
# with the cheapest call the provider offers (Anthropic Haiku ping, OpenAI
# models.list, Together models.list).

import anthropic
import openai
import together

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
OPENAI_API_KEY = getpass.getpass("OPENAI_API_KEY: ").strip()
ANTHROPIC_API_KEY = getpass.getpass("ANTHROPIC_API_KEY: ").strip()
TOGETHER_API_KEY = getpass.getpass("TOGETHER_API_KEY: ").strip()

if not (OPENAI_API_KEY and ANTHROPIC_API_KEY and TOGETHER_API_KEY):
    print("All three provider keys are required for this lab. Re-run this cell.")
    raise SystemExit(1)

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY
os.environ["TOGETHER_API_KEY"] = TOGETHER_API_KEY

try:
    openai_client = openai.OpenAI(api_key=OPENAI_API_KEY)
    openai_client.models.list()
    print("OpenAI credentials validated.")
except Exception as exc:
    print(f"OpenAI key rejected: {exc!r}")
    raise SystemExit(1)

try:
    anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    _ping = anthropic_client.messages.create(
        model=PRICE_MAP["anthropic"]["model"],
        max_tokens=8,
        messages=[{"role": "user", "content": "reply with: ok"}],
    )
    print(f"Anthropic credentials validated. Haiku replied: {_ping.content[0].text!r}")
except Exception as exc:
    print(f"Anthropic key rejected: {exc!r}")
    raise SystemExit(1)

try:
    together_client = together.Together(api_key=TOGETHER_API_KEY)
    list(together_client.models.list())[:1]
    print("Together credentials validated.")
except Exception as exc:
    print(f"Together key rejected: {exc!r}")
    raise SystemExit(1)

register(
    session_token=session_token,
    lab_id=LAB_ID,
    credentials={"providers": list(PROVIDER_ORDER)},
)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, inline adapter, atexit safety net, orphan scan.
# All resources for this lab are local files in /content; no external state.
# The orphan scan checks the two known paths for leftover state.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="local_file",
        id=TRACES_PATH,
        region="local",
        cli_delete_command=f"rm -f {TRACES_PATH}",
    ),
    CleanupResource(
        type="local_file",
        id=BREAKER_PATH,
        region="local",
        cli_delete_command=f"rm -f {BREAKER_PATH}",
    ),
]


class GatewayLabCleanupAdapter:
    """Local-file cleanup for the gateway lab. Removes trace + breaker JSON.

    TODO: a labrun-checks `local` adapter is not in 0.7.0; inline bridge here.
    """

    def delete_resource(self, credentials, resource):
        if resource.type == "local_file":
            try:
                os.remove(resource.id)
            except FileNotFoundError:
                return

    def describe_resource(self, credentials, resource):
        if resource.type == "local_file":
            return os.path.exists(resource.id)
        return False

    def tag_scan(self, credentials, lab_slug, region):
        # No external tag store; the orphan check below covers the local paths.
        return []


CLEANUP_ADAPTER = GatewayLabCleanupAdapter()


def _orphan_scan():
    found = []
    for p in (TRACES_PATH, BREAKER_PATH):
        if os.path.exists(p):
            found.append(p)
    return found


_orphans = _orphan_scan()
if _orphans:
    print("Orphan scan found leftover files from a prior session:")
    for p in _orphans:
        print(f"  {p}")
    print("Run this cell after deleting them, or re-run the cleanup cell from a prior session.")
    raise SystemExit(1)


def _on_exit_cleanup():
    try:
        for r in CLEANUP_MANIFEST:
            try:
                CLEANUP_ADAPTER.delete_resource({}, r)
            except Exception:
                pass
    except Exception:
        pass


atexit.register(_on_exit_cleanup)
print("Manifest registered. Orphan scan clean.")

## Task 1: Wrap each provider in a uniform Provider class

Three different SDKs, three different response shapes. The gateway treats them all as one interface so the failover loop does not branch on provider name. Build `Provider` with subclasses `OpenAIProvider`, `AnthropicProvider`, `TogetherProvider`, each exposing `generate(prompt) -> dict` returning fields `text`, `cost_cents`, `latency_ms`, `input_tokens`, `output_tokens`.

Then run a 30-request smoke test that round-robins the three providers and records each response into `gateway_traces.json`. Each provider should land at least 5 successes.

In [ ]:
# Task 1: Provider classes plus the round-robin smoke test.

class Provider:
    """Uniform interface. Subclasses implement _call() and _tokens()."""

    name = "base"

    def generate(self, prompt):
        start = time.perf_counter()
        text, in_tok, out_tok = self._call(prompt)
        latency_ms = int((time.perf_counter() - start) * 1000)
        price = PRICE_MAP[self.name]
        cost_cents = ((in_tok * price["in_per_m"]) + (out_tok * price["out_per_m"])) / 1_000_000 * 100
        return {
            "provider": self.name,
            "model": price["model"],
            "text": text,
            "input_tokens": in_tok,
            "output_tokens": out_tok,
            "cost_cents": round(cost_cents, 6),
            "latency_ms": latency_ms,
        }

    def _call(self, prompt):
        raise NotImplementedError


class OpenAIProvider(Provider):
    name = "openai"

    def _call(self, prompt):
        # YOUR CODE: call openai_client.chat.completions.create with model
        # PRICE_MAP["openai"]["model"], a single user message containing
        # prompt, and max_tokens=64. Return (text, input_tokens, output_tokens)
        # extracted from resp.choices[0].message.content and resp.usage.
        raise NotImplementedError("Replace with the OpenAI SDK call.")


class AnthropicProvider(Provider):
    name = "anthropic"

    def _call(self, prompt):
        # YOUR CODE: call anthropic_client.messages.create with model
        # PRICE_MAP["anthropic"]["model"], a single user message, max_tokens=64.
        # Return (text, usage.input_tokens, usage.output_tokens).
        raise NotImplementedError("Replace with the Anthropic SDK call.")


class TogetherProvider(Provider):
    name = "together"

    def _call(self, prompt):
        # YOUR CODE: call together_client.chat.completions.create with model
        # PRICE_MAP["together"]["model"], a single user message, max_tokens=64.
        # Return (text, usage.prompt_tokens, usage.completion_tokens).
        raise NotImplementedError("Replace with the Together SDK call.")


PROVIDERS = {
    "openai": OpenAIProvider(),
    "anthropic": AnthropicProvider(),
    "together": TogetherProvider(),
}


def _append_trace(record):
    rows = []
    if os.path.exists(TRACES_PATH):
        try:
            rows = json.loads(open(TRACES_PATH).read())
        except Exception:
            rows = []
    rows.append(record)
    open(TRACES_PATH, "w").write(json.dumps(rows))


SMOKE_PROMPTS = [
    "Say the capital of France in one word.",
    "Say a primary color in one word.",
    "Say the largest planet in one word.",
] * 10  # 30 prompts


def smoke_test():
    for i, prompt in enumerate(SMOKE_PROMPTS):
        provider = PROVIDERS[PROVIDER_ORDER[i % 3]]
        try:
            rec = provider.generate(prompt)
            rec["status"] = "ok"
        except Exception as exc:
            rec = {
                "provider": provider.name,
                "model": PRICE_MAP[provider.name]["model"],
                "status": "error",
                "error": repr(exc),
                "latency_ms": 0,
                "cost_cents": 0.0,
                "input_tokens": 0,
                "output_tokens": 0,
            }
        _append_trace(rec)
    print(f"Smoke test complete. Traces written to {TRACES_PATH}.")


smoke_test()

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: each of the three providers landed >= 5 successes in the
# smoke test recorded in gateway_traces.json.


def checkpoint_1(session):
    if not os.path.exists(TRACES_PATH):
        return CheckpointResult(
            status="fail",
            error_reason=f"{TRACES_PATH} does not exist; the smoke test did not run.",
        )
    rows = json.loads(open(TRACES_PATH).read())
    per_provider_ok = {p: 0 for p in PROVIDER_ORDER}
    for r in rows:
        if r.get("status") == "ok" and r.get("provider") in per_provider_ok:
            per_provider_ok[r["provider"]] += 1
    for p in PROVIDER_ORDER:
        if per_provider_ok[p] < 5:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Provider {p!r} landed {per_provider_ok[p]} successes; need >= 5. "
                    f"Most likely the Provider._call() for {p} still raises NotImplementedError."
                ),
            )
    return CheckpointResult(status="pass")


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

Each SDK has its own response shape. Read the docs once per provider before guessing the field names. The token counts live on `resp.usage` for OpenAI and Together (`prompt_tokens`/`completion_tokens`), and on `resp.usage` for Anthropic (`input_tokens`/`output_tokens`).

</details>

<details><summary>Hint 2 (direction)</summary>

OpenAI: the chat completions API returns `resp.choices[0].message.content` and `resp.usage.prompt_tokens` / `resp.usage.completion_tokens`. Anthropic's messages API returns `resp.content[0].text` (the content is a list of blocks; text lives on the first block) and `resp.usage.input_tokens` / `resp.usage.output_tokens`. Together mirrors OpenAI's chat completions shape.

The trace JSON file is a list of dicts. Append, do not overwrite. The validator counts per-provider successes by reading every row.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working `_call` for each provider:

```python
class OpenAIProvider(Provider):
    name = "openai"
    def _call(self, prompt):
        resp = openai_client.chat.completions.create(
            model=PRICE_MAP["openai"]["model"],
            messages=[{"role": "user", "content": prompt}],
            max_tokens=64,
        )
        return (
            resp.choices[0].message.content,
            resp.usage.prompt_tokens,
            resp.usage.completion_tokens,
        )


class AnthropicProvider(Provider):
    name = "anthropic"
    def _call(self, prompt):
        resp = anthropic_client.messages.create(
            model=PRICE_MAP["anthropic"]["model"],
            messages=[{"role": "user", "content": prompt}],
            max_tokens=64,
        )
        return (resp.content[0].text, resp.usage.input_tokens, resp.usage.output_tokens)


class TogetherProvider(Provider):
    name = "together"
    def _call(self, prompt):
        resp = together_client.chat.completions.create(
            model=PRICE_MAP["together"]["model"],
            messages=[{"role": "user", "content": prompt}],
            max_tokens=64,
        )
        return (
            resp.choices[0].message.content,
            resp.usage.prompt_tokens,
            resp.usage.completion_tokens,
        )
```

</details>

**Wallet check.** 30 smoke calls split across three providers. OpenAI's GPT-4o-mini at $0.15/1M input runs about a tenth of a penny per call; Anthropic Haiku at $0.80/1M is ~5x more; Together Llama is ~$0.20/1M. Total damage so far: roughly 5 cents. Your morning coffee still wins.

## Task 2: Circuit breaker that opens on persistent failure

The lab ships a `sabotage_provider(name, fail_rate)` helper that monkey-patches a provider so `_call` raises `RuntimeError("simulated 5xx")` at the requested rate. Use it to drive a stream of failures into OpenAI; the breaker must open after `BREAKER_FAILURE_THRESHOLD` consecutive failures and skip OpenAI until the cooldown expires.

The breaker state persists to `circuit_breaker_state.json` so a kernel restart does not blow up the state machine.

In [ ]:
# Task 2: CircuitBreaker plus the sabotage helper.

_breaker_lock = Lock()


class CircuitBreaker:
    """Per-provider breaker. Tracks consecutive failures; opens at threshold.

    State is one dict per provider: {"failures": int, "opened_at": float | None}.
    """

    def __init__(self, threshold, cooldown_sec, state_path):
        self.threshold = threshold
        self.cooldown_sec = cooldown_sec
        self.state_path = state_path
        self.state = {p: {"failures": 0, "opened_at": None} for p in PROVIDER_ORDER}
        self._load()

    def _load(self):
        if os.path.exists(self.state_path):
            try:
                self.state.update(json.loads(open(self.state_path).read()))
            except Exception:
                pass

    def _save(self):
        with _breaker_lock:
            open(self.state_path, "w").write(json.dumps(self.state))

    def is_open(self, name):
        # YOUR CODE: read self.state[name]. If opened_at is None, return False.
        # If opened_at is set and (time.time() - opened_at) < cooldown_sec,
        # return True. Otherwise reset failures to 0, opened_at to None, save,
        # return False (cooldown elapsed; back to half-open).
        raise NotImplementedError("Replace with the open/half-open logic.")

    def record_success(self, name):
        # YOUR CODE: reset failures to 0 and opened_at to None, then save.
        raise NotImplementedError

    def record_failure(self, name):
        # YOUR CODE: increment failures by 1. If failures >= threshold and
        # opened_at is None, set opened_at to time.time(). Save.
        raise NotImplementedError


breaker = CircuitBreaker(
    threshold=BREAKER_FAILURE_THRESHOLD,
    cooldown_sec=BREAKER_COOLDOWN_SEC,
    state_path=BREAKER_PATH,
)


def sabotage_provider(name, fail_rate):
    """Monkey-patch a Provider so _call raises at the given rate."""
    p = PROVIDERS[name]
    real_call = p._call

    def patched(prompt):
        if random.random() < fail_rate:
            raise RuntimeError(f"simulated 5xx on {name}")
        return real_call(prompt)

    p._call = patched
    p._original_call = real_call


def restore_provider(name):
    p = PROVIDERS[name]
    if hasattr(p, "_original_call"):
        p._call = p._original_call
        del p._original_call


# Drive 10 failures into OpenAI to open the breaker.
sabotage_provider("openai", fail_rate=1.0)
for _ in range(10):
    try:
        PROVIDERS["openai"].generate("ping")
        breaker.record_success("openai")
    except Exception:
        breaker.record_failure("openai")

print(f"OpenAI breaker state: {breaker.state['openai']}")
print(f"OpenAI is_open: {breaker.is_open('openai')}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: OpenAI breaker is_open after the sabotage drove >= threshold
# consecutive failures, and the persisted state file reflects opened_at.


def checkpoint_2(session):
    try:
        is_open = breaker.is_open("openai")
    except Exception as exc:
        return CheckpointResult(
            status="error",
            error_reason=f"CircuitBreaker.is_open raised {exc!r}; the open-state logic is incomplete.",
        )
    if not is_open:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"OpenAI breaker is_open=False after 10 forced failures. "
                f"Either failures count is not incrementing or threshold check is off. "
                f"Current state: {breaker.state.get('openai')!r}."
            ),
        )
    if not os.path.exists(BREAKER_PATH):
        return CheckpointResult(
            status="fail",
            error_reason=f"{BREAKER_PATH} not written; record_failure must call self._save() so state survives kernel restarts.",
        )
    persisted = json.loads(open(BREAKER_PATH).read())
    if not persisted.get("openai", {}).get("opened_at"):
        return CheckpointResult(
            status="fail",
            error_reason="opened_at is not persisted; record_failure should set opened_at when failures crosses threshold.",
        )
    return CheckpointResult(status="pass")


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

A breaker is a tiny state machine: `closed` (failures < threshold), `open` (failures >= threshold, recent), `half-open` (open but cooldown elapsed; next call is a probe). For this lab, half-open collapses into "next call closes again on success or reopens on failure"; you do not need a dedicated half-open state, only the transition logic in `is_open`.

</details>

<details><summary>Hint 2 (direction)</summary>

`record_failure` increments the counter, then checks if it just crossed the threshold and sets `opened_at = time.time()` if so. `is_open` returns False if `opened_at is None`. Otherwise it compares wall-clock now against `opened_at + cooldown`; inside the cooldown window, return True; outside it, treat the breaker as auto-reset (clear `failures`, clear `opened_at`, save) and return False so the next call is the probe.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Full implementation:

```python
def is_open(self, name):
    s = self.state[name]
    if s["opened_at"] is None:
        return False
    if (time.time() - s["opened_at"]) < self.cooldown_sec:
        return True
    s["failures"] = 0
    s["opened_at"] = None
    self._save()
    return False

def record_success(self, name):
    s = self.state[name]
    s["failures"] = 0
    s["opened_at"] = None
    self._save()

def record_failure(self, name):
    s = self.state[name]
    s["failures"] += 1
    if s["failures"] >= self.threshold and s["opened_at"] is None:
        s["opened_at"] = time.time()
    self._save()
```

</details>

**Wallet check.** Ten extra calls to test the breaker, plus the 30 from Task 1. Still under 10 cents. The sabotage helper monkey-patched the call itself, so the failures did not actually hit the network beyond the smoke test.

## Task 3: Per-request cost attribution log

The trace records already capture `cost_cents`, `input_tokens`, `output_tokens` and `provider` because `Provider.generate` computed them. This task is the persistence + invariant: every trace row written by the gateway must have those four fields plus `model`, and the cost in cents must equal what the price map says given the recorded token counts.

In [ ]:
# Task 3: a small audit pass over the trace file confirming every row has
# the cost fields and that recorded cost matches the price-map math.

restore_provider("openai")
breaker.record_success("openai")  # reset before the audit
breaker._save()


def audit_trace_costs():
    rows = json.loads(open(TRACES_PATH).read()) if os.path.exists(TRACES_PATH) else []
    missing = []
    mismatched = []
    for row in rows:
        if row.get("status") != "ok":
            continue
        # YOUR CODE: confirm row has keys provider, model, input_tokens,
        # output_tokens, cost_cents. Append the missing keys to `missing`
        # if any are absent.
        # YOUR CODE: recompute the expected cost from PRICE_MAP and the
        # row's input_tokens/output_tokens; if abs(expected - actual) > 1e-6,
        # append a (provider, expected, actual) tuple to mismatched.
        pass
    return missing, mismatched


missing, mismatched = audit_trace_costs()
print(f"Rows audited. Missing-field rows: {len(missing)}. Mismatched-cost rows: {len(mismatched)}.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: every successful trace row has the five cost-attribution
# fields, and recorded cost_cents matches the PRICE_MAP recomputation.


def checkpoint_3(session):
    if not os.path.exists(TRACES_PATH):
        return CheckpointResult(status="fail", error_reason=f"{TRACES_PATH} missing.")
    rows = json.loads(open(TRACES_PATH).read())
    ok_rows = [r for r in rows if r.get("status") == "ok"]
    if len(ok_rows) < 15:
        return CheckpointResult(
            status="fail",
            error_reason=f"Only {len(ok_rows)} ok rows in the trace; the smoke test should produce >= 15.",
        )
    required = {"provider", "model", "input_tokens", "output_tokens", "cost_cents"}
    for row in ok_rows:
        missing_keys = required - set(row.keys())
        if missing_keys:
            return CheckpointResult(
                status="fail",
                error_reason=f"Row missing required cost-attribution keys: {missing_keys}.",
            )
        price = PRICE_MAP.get(row["provider"])
        if price is None:
            continue
        expected = ((row["input_tokens"] * price["in_per_m"]) + (row["output_tokens"] * price["out_per_m"])) / 1_000_000 * 100
        if abs(expected - row["cost_cents"]) > 1e-4:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Cost mismatch on a {row['provider']} row: expected {expected:.6f} cents, "
                    f"got {row['cost_cents']:.6f}. Check the PRICE_MAP math in Provider.generate()."
                ),
            )
    return CheckpointResult(status="pass")


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

The fields are already in the trace because `Provider.generate` wrote them. The audit is a sanity check; if it flags missing fields, the cause is almost always a manual override of the dict in your `_call` or an out-of-order edit. The cost recomputation is the price map applied to the recorded token counts.

</details>

<details><summary>Hint 2 (direction)</summary>

For each ok row, run:

```
expected = ((row["input_tokens"] * price["in_per_m"]) + (row["output_tokens"] * price["out_per_m"])) / 1_000_000 * 100
```

then compare with `row["cost_cents"]` within a tolerance of 1e-6 cents. If you find drift, the most likely cause is a stale per-1M rate or a missing divide-by-1M.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Full audit loop body:

```python
required = {"provider", "model", "input_tokens", "output_tokens", "cost_cents"}
for row in rows:
    if row.get("status") != "ok":
        continue
    miss = required - set(row.keys())
    if miss:
        missing.append((row.get("provider"), miss))
        continue
    price = PRICE_MAP.get(row["provider"])
    if price is None:
        continue
    expected = ((row["input_tokens"] * price["in_per_m"]) + (row["output_tokens"] * price["out_per_m"])) / 1_000_000 * 100
    if abs(expected - row["cost_cents"]) > 1e-6:
        mismatched.append((row["provider"], expected, row["cost_cents"]))
```

</details>

**Wallet check.** Audit is free (zero new LLM calls). Cumulative spend is still pennies. The cost-per-request log is the production artifact you would forward to Datadog or PostHog; the lab keeps it as JSON to stay dependency-light.

## Task 4: 100-request outage simulation

Final task. Sabotage all three providers at different rates (OpenAI 100%, Anthropic 50%, Together 0%), then run a 100-request load test through the full gateway: failover across providers in order, with the circuit breaker active and a 3-second per-provider deadline. The gateway should land >= 95% success by falling through to Together when the upstream two reject.

In [ ]:
# Task 4: full gateway call with timeout+failover+breaker, then a 100-request
# load test under simulated multi-provider outage.

breaker = CircuitBreaker(BREAKER_FAILURE_THRESHOLD, BREAKER_COOLDOWN_SEC, BREAKER_PATH)


@contextmanager
def _deadline(seconds):
    """Cheap soft-deadline: just records start time so the caller can check.
    Real production code uses signal.SIGALRM or asyncio.wait_for; both are
    awkward in Colab kernels because signals do not work in threads. The
    soft check below is fine for this lab because the slow path is the
    requests timeout inside each SDK, which we set on the SDK call.
    """
    start = time.perf_counter()

    class _Deadline:
        def remaining(self):
            return seconds - (time.perf_counter() - start)

        def expired(self):
            return self.remaining() <= 0

    yield _Deadline()


def gateway_call(prompt):
    """Try each provider in order, respecting the breaker and the deadline."""
    last_err = None
    for name in PROVIDER_ORDER:
        if breaker.is_open(name):
            continue
        # YOUR CODE: call PROVIDERS[name].generate(prompt). On success, call
        # breaker.record_success(name), append the trace, and return the
        # response dict with an added "gateway_path": [name] field. On
        # exception, call breaker.record_failure(name), save the exception
        # to last_err, and continue to the next provider in the loop.
        raise NotImplementedError("Replace with the failover loop body.")
    return {"status": "all_failed", "error": repr(last_err), "gateway_path": list(PROVIDER_ORDER)}


# Sabotage all three at the brief's rates.
sabotage_provider("openai", 1.0)
sabotage_provider("anthropic", 0.5)
restore_provider("together")

LOAD_PROMPTS = ["Reply with a single word: ok"] * 100
load_results = []
for prompt in LOAD_PROMPTS:
    rec = gateway_call(prompt)
    rec["prompt"] = prompt[:40]
    load_results.append(rec)
    # Append every load-test row to the trace file too, so the checkpoint can
    # query a single source of truth.
    _append_trace({**rec, "phase": "load_test"})

succ = sum(1 for r in load_results if r.get("status") not in ("all_failed", "error"))
print(f"Load test complete. Successes: {succ}/100 ({succ}%).")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: load test success rate >= 0.95 (>= 95 of 100 ok) when the
# trace file's load_test phase rows are aggregated.


def checkpoint_4(session):
    if not os.path.exists(TRACES_PATH):
        return CheckpointResult(status="fail", error_reason=f"{TRACES_PATH} missing.")
    rows = json.loads(open(TRACES_PATH).read())
    load_rows = [r for r in rows if r.get("phase") == "load_test"]
    if len(load_rows) < 100:
        return CheckpointResult(
            status="fail",
            error_reason=f"Found {len(load_rows)} load_test rows; the load test should produce 100.",
        )
    succ = sum(1 for r in load_rows if r.get("status") not in ("all_failed", "error", None))
    rate = succ / len(load_rows)
    if rate < 0.95:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Success rate {rate:.2%} < 95%. Most common cause: gateway aborts on first "
                f"exception instead of falling through; or the breaker opens Together too."
            ),
        )
    return CheckpointResult(status="pass")


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

The failover loop is in priority order: try OpenAI, then Anthropic, then Together. Each provider gets one attempt per request. The breaker is the short-circuit: if it is open, you skip without even trying. Catch the exception, log it, move on.

</details>

<details><summary>Hint 2 (direction)</summary>

```
for name in PROVIDER_ORDER:
    if breaker.is_open(name):
        continue
    try:
        rec = PROVIDERS[name].generate(prompt)
        breaker.record_success(name)
        rec["status"] = "ok"
        rec["gateway_path"] = [name]
        return rec
    except Exception as exc:
        breaker.record_failure(name)
        last_err = exc
return {"status": "all_failed", "error": repr(last_err), "gateway_path": list(PROVIDER_ORDER)}
```

The per-provider deadline matters in production (real SDK timeouts), but for this lab the sabotage raises synchronously, so the loop falls through fast. In production you would set `openai_client = openai.OpenAI(timeout=3.0)` or pass `request_options={"timeout": 3.0}` per call.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete `gateway_call`:

```python
def gateway_call(prompt):
    last_err = None
    for name in PROVIDER_ORDER:
        if breaker.is_open(name):
            continue
        try:
            rec = PROVIDERS[name].generate(prompt)
            breaker.record_success(name)
            rec["status"] = "ok"
            rec["gateway_path"] = [name]
            return rec
        except Exception as exc:
            breaker.record_failure(name)
            last_err = exc
            continue
    return {"status": "all_failed", "error": repr(last_err), "gateway_path": list(PROVIDER_ORDER)}
```

</details>

**Wallet check.** Outage simulation: 100 requests, OpenAI sabotaged to 100% fail (no real network), Anthropic sabotaged to 50% (50 partial network hits to Haiku), Together at 0% (the remaining 95+ requests land on Llama). Spent so far across the whole lab: roughly 80 cents to a dollar.

## Cleanup

Local-file teardown only. The gateway, the breaker state, and the trace log all live in /content. Run the cell. Re-running is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup. Two local files. No external resources.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, the gateway lab left local state behind. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: about $1.00.** Mostly Llama at $0.20/1M and a sprinkle of Haiku at $0.80/1M during the partial-sabotage half of the load test. OpenAI fully sabotaged contributed zero spend. Local files cleared; the gateway lives only in your kernel.

## Reflection

These are not graded. They are for you.

1. Your gateway routes by priority order (OpenAI first, then Anthropic, then Together). The brief mentions latency p95 routing in normal operation: in production, p95 drifts over time as providers tune their fleets. What is the simplest re-evaluation pattern that does not require a separate analytics service, given you already write per-request latency to the trace log?

2. Each request emits a cost cell. The product owner wants daily and weekly cost reports per feature (not per request). What is the simplest aggregation that does not need a separate analytics DB, given the trace file is JSON-line-style append-only?

## Exam-Style Practice

**Question 1 (circuit breaker semantics):**

A model gateway routes between OpenAI and Anthropic. OpenAI returns 503 for 3 consecutive requests. The circuit breaker should:

A. Permanently disable OpenAI.
B. Open for a cool-down period (e.g., 30 seconds), then attempt a probe request to check recovery.
C. Retry immediately.
D. Fall back to Anthropic but keep sending all traffic to OpenAI in parallel.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: permanent disable removes self-healing; outages are usually transient.
- B is correct: half-open state is the canonical pattern. Open for cooldown, then a single probe; success closes the breaker, another failure reopens with a fresh cooldown.
- C is wrong: immediate retry is what triggered the open in the first place; the cooldown is the back-off.
- D is wrong: parallel duplication wastes spend and doubles upstream pressure during an outage.

</details>

**Question 2 (timeout discipline):**

A gateway tries three providers sequentially. The team observes that overall request latency exceeds 30 seconds during partial outages. Which single change has the highest leverage?

A. Switch to async/await so the calls overlap.
B. Set a tight per-provider request timeout (for example 3 seconds) so a slow provider does not eat the latency budget.
C. Use a larger model on the primary so it succeeds more often.
D. Add a retry on each provider before moving on.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: overlapping the calls (hedging) saves latency but doubles or triples cost during normal operation; a timeout is cheaper.
- B is correct: a 3 second per-provider deadline caps the failover budget at roughly N x 3 seconds across N providers; a missing timeout lets a hung TCP socket consume the full request budget.
- C is wrong: a larger model is slower per success and does not fix the failure path.
- D is wrong: retries multiply latency during an outage; you want fail-fast plus failover, not retry-and-fall-through.

</details>

**Question 3 (cost attribution):**

A team's monthly LLM bill jumped 3x. The gateway logs per-request `cost_cents`, `provider`, `model`, `input_tokens`, `output_tokens`. Which is the highest-leverage first query?

A. Group by provider and compare to last month's per-provider totals.
B. Group by user_id to find the heaviest user.
C. Group by hour to find a traffic spike.
D. Group by model to confirm no one upgraded to a more expensive model.

<details><summary>Show answer</summary>

**Correct: D.**

- A is reasonable but slow: provider distribution shifts gradually; a 3x jump in a month is rarely a provider-mix change.
- B is wrong for a 3x jump: a single user 3xing usage rarely 3xs the bill unless that user was the bulk of traffic.
- C is rarely the cause of a 3x monthly jump: traffic spikes are usually hourly or daily, not monthly.
- D is correct: a model upgrade (Haiku to Sonnet, GPT-4o-mini to GPT-4o) is the most common cause of an order-of-magnitude monthly cost jump. The model field on each row is the fastest signal.

</details>